In [1]:
# NetGuardian - Network Intrusion Detection System
# Optimized for Kaggle environment with large dataset processing

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from imblearn.under_sampling import RandomUnderSampler
import joblib
import logging
import os
import gc
from tqdm import tqdm
import time

In [2]:
# Part 1: Setup and Configuration
# -------------------------------

def setup_logging():
    """Configure logging for the application."""
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler("netguardian.log"),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

logger = setup_logging()
logger.info("NetGuardian starting - Kaggle optimized version")

# Configure memory usage reduction techniques
def reduce_memory_usage(df):
    """Reduce memory usage of a DataFrame."""
    start_mem = df.memory_usage().sum() / 1024**2
    logger.info(f"Memory usage before optimization: {start_mem:.2f} MB")
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    
    end_mem = df.memory_usage().sum() / 1024**2
    logger.info(f"Memory usage after optimization: {end_mem:.2f} MB")
    logger.info(f"Reduced by {100 * (start_mem - end_mem) / start_mem:.1f}%")
    return df

In [3]:
# Part 2: Data Loading and Preprocessing
# -------------------------------------

def load_data_in_chunks(file_path, chunksize=500000):
    """Load large dataset in chunks to prevent memory issues."""
    logger.info(f"Loading data in chunks from {file_path}")
    chunks = []
    
    # Get total number of rows for progress tracking
    total_rows = sum(1 for _ in open(file_path, 'r')) - 1  # Subtract header row
    
    with tqdm(total=total_rows) as pbar:
        for chunk in pd.read_csv(file_path, chunksize=chunksize):
            chunk = reduce_memory_usage(chunk)
            chunks.append(chunk)
            pbar.update(len(chunk))
            
    df = pd.concat(chunks, ignore_index=True)
    logger.info(f"Data loaded successfully: {df.shape[0]} rows and {df.shape[1]} columns")
    return df

def preprocess_data(df, label_column=' Label'):
    """Preprocess the dataset by handling missing values and encoding the target."""
    logger.info("Preprocessing data...")
    
    # Handle missing or infinite values
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Count NaN values before dropping
    nan_count = df.isna().sum().sum()
    logger.info(f"Found {nan_count} NaN values")
    
    # Drop rows with NaN values
    df.dropna(inplace=True)
    logger.info(f"Shape after dropping NaN values: {df.shape}")

    # Verify label column exists
    if label_column not in df.columns:
        raise KeyError(f"The column '{label_column}' was not found in the dataset.")
    
    # Binary encoding of target: BENIGN=0, all attacks=1
    df[label_column] = df[label_column].apply(lambda x: 0 if x == 'BENIGN' else 1)
    
    # Log class distribution
    class_counts = df[label_column].value_counts()
    logger.info(f"Class distribution: {class_counts.to_dict()}")
    logger.info(f"Class balance ratio (benign:attack): {class_counts[0]/class_counts[1]:.2f}:1")
    
    return df

In [4]:
# Part 3: Data Splitting and Sampling
# ----------------------------------

def prepare_training_data(df, label_column=' Label', sample_size=100000, test_size=0.2):
    """Split data and prepare samples for training."""
    logger.info("Preparing training and testing datasets...")
    
    # Separate features and target
    if 'source_file' in df.columns:
        X = df.drop(columns=[label_column, 'source_file'])
    else:
        X = df.drop(columns=[label_column])
        
    y = df[label_column]
    
    # First split: Create a small test set
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, stratify=y, test_size=test_size, random_state=42
    )
    
    # Free memory
    del X, y, df
    gc.collect()
    
    # Second split: Create a manageable training set
    # Calculate balanced sample sizes for each class
    benign_count = sum(y_train_full == 0)
    attack_count = sum(y_train_full == 1)
    
    # Aim for roughly balanced classes in sample
    benign_sample = min(int(sample_size/2), benign_count)
    attack_sample = min(int(sample_size/2), attack_count)
    
    logger.info(f"Sampling {benign_sample} benign and {attack_sample} attack records")
    
    # Sample benign records
    benign_indices = y_train_full[y_train_full == 0].index
    benign_sample_indices = np.random.choice(benign_indices, benign_sample, replace=False)
    
    # Sample attack records
    attack_indices = y_train_full[y_train_full == 1].index
    attack_sample_indices = np.random.choice(attack_indices, attack_sample, replace=False)
    
    # Combine samples
    sample_indices = np.concatenate([benign_sample_indices, attack_sample_indices])
    X_train = X_train_full.loc[sample_indices]
    y_train = y_train_full.loc[sample_indices]
    
    # Free more memory
    del X_train_full, y_train_full, benign_indices, attack_indices
    gc.collect()
    
    logger.info(f"Training sample: {X_train.shape[0]} rows, {X_train.shape[1]} features")
    logger.info(f"Test set: {X_test.shape[0]} rows, {X_test.shape[1]} features")
    
    return X_train, X_test, y_train, y_test

In [5]:
# Part 4: Feature Scaling and Balancing
# -----------------------------------

def scale_features(X_train, X_test):
    """Scale features using StandardScaler."""
    logger.info("Scaling features...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Save the scaler for later use
    os.makedirs('./models', exist_ok=True)
    scaler_path = './models/scaler.pkl'
    joblib.dump(scaler, scaler_path)
    logger.info(f"Scaler saved at: {scaler_path}")
    
    return X_train_scaled, X_test_scaled, scaler

def balance_data(X_train_scaled, y_train, strategy='undersample'):
    """Balance the dataset using sampling techniques."""
    if strategy == 'undersample':
        logger.info("Balancing data with RandomUnderSampler...")
        # Using undersampling for large datasets
        sampler = RandomUnderSampler(random_state=42)
        X_train_balanced, y_train_balanced = sampler.fit_resample(X_train_scaled, y_train)
        
    logger.info(f"Balanced data shape: {X_train_balanced.shape}")
    class_counts = np.bincount(y_train_balanced)
    logger.info(f"Balanced class distribution: {class_counts}")
    
    return X_train_balanced, y_train_balanced

In [6]:
# Part 5: Model Training
# --------------------

def train_model(X_train, y_train, X_test, y_test):
    """Train XGBoost model with RandomizedSearchCV."""
    logger.info("Setting up XGBoost model training...")
    
    # Calculate weight for imbalanced classes
    scale_pos_weight = sum(y_train == 0) / sum(y_train == 1)
    logger.info(f"scale_pos_weight: {scale_pos_weight:.4f}")
    
    # Configure XGBoost with parameters suitable for Kaggle
    xgb_model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        use_label_encoder=False,
        tree_method='hist',  # Faster algorithm
        random_state=42
    )
    
    # Parameter grid for RandomizedSearchCV (more efficient than GridSearchCV)
    param_dist = {
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.8, 0.9],
        'colsample_bytree': [0.8, 0.9],
        'scale_pos_weight': [scale_pos_weight],
        'n_estimators': [100, 200]
    }
    
    # Use RandomizedSearchCV to save computation time
    logger.info("Starting RandomizedSearchCV...")
    start_time = time.time()
    
    search = RandomizedSearchCV(
        estimator=xgb_model,
        param_distributions=param_dist,
        scoring='f1',
        n_iter=10,  # Try 10 combinations instead of all combinations
        cv=3,
        verbose=1,
        n_jobs=-1,
        random_state=42
    )
    
    search.fit(X_train, y_train, 
              eval_set=[(X_train, y_train), (X_test, y_test)],
              early_stopping_rounds=10,
              verbose=0)
    
    training_time = time.time() - start_time
    logger.info(f"Training completed in {training_time:.2f} seconds")
    logger.info(f"Best parameters: {search.best_params_}")
    
    return search.best_estimator_

In [7]:
# Part 6: Model Evaluation
# ----------------------

def evaluate_model(model, X_test, y_test):
    """Evaluate the trained model on test data."""
    logger.info("Evaluating model on test set...")
    
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred)
    
    # Log results
    logger.info(f"Accuracy: {accuracy:.4f}")
    logger.info(f"F1 Score: {f1:.4f}")
    logger.info(f"Confusion Matrix:\n{conf_matrix}")
    logger.info(f"Classification Report:\n{class_report}")
    
    # Print results for notebook output
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Confusion Matrix:")
    print(conf_matrix)
    print("\nClassification Report:")
    print(class_report)
    
    return accuracy, f1, conf_matrix, class_report

def visualize_results(model, feature_names):
    """Create and save visualizations of model results."""
    logger.info("Creating visualizations...")
    os.makedirs('./models', exist_ok=True)
    
    # Plot feature importance
    plt.figure(figsize=(12, 8))
    xgb.plot_importance(model, max_num_features=20, importance_type='weight')
    plt.title("Feature Importance (Weight)")
    plt.tight_layout()
    plt.savefig('./models/feature_importance_weight.png')
    logger.info("Saved feature importance (weight) plot")
    plt.close()
    
    # Plot feature importance by gain
    plt.figure(figsize=(12, 8))
    xgb.plot_importance(model, max_num_features=20, importance_type='gain')
    plt.title("Feature Importance (Gain)")
    plt.tight_layout()
    plt.savefig('./models/feature_importance_gain.png')
    logger.info("Saved feature importance (gain) plot")
    plt.close()
    
    # Get and save feature importance scores
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Weight': model.feature_importances_
    }).sort_values(by='Weight', ascending=False)
    
    importance_df.to_csv('./models/feature_importance.csv', index=False)
    logger.info("Saved feature importance scores to CSV")


In [8]:
# Part 7: Model Persistence
# -----------------------

def save_model(model, filename='xgboost_netguardian_model.pkl'):
    """Save the trained model to disk."""
    os.makedirs('./models', exist_ok=True)
    model_path = os.path.join('./models', filename)
    joblib.dump(model, model_path)
    logger.info(f"Model saved to {model_path}")
    
    # Save model in XGBoost native format too
    xgb_native_path = os.path.join('./models', 'model.xgb')
    model.save_model(xgb_native_path)
    logger.info(f"Model also saved in XGBoost native format to {xgb_native_path}")
    
    return model_path

In [9]:
# Part 8: Main Execution
# --------------------

def main():
    """Main execution function."""
    start_time = time.time()
    logger.info("Starting NetGuardian training pipeline")
    
    # Define the file path for the dataset
    cleaned_file_path = '/kaggle/input/clean-dataset/clean_dataset.csv'
    
    try:
        # Load data in chunks
        df = load_data_in_chunks(cleaned_file_path)
        
        # Preprocess data
        df = preprocess_data(df)
        
        # Split and sample data
        X_train, X_test, y_train, y_test = prepare_training_data(
            df, sample_size=100000, test_size=0.2
        )
        
        # Free memory
        del df
        gc.collect()
        
        # Scale features
        X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)
        
        # Balance data
        X_train_balanced, y_train_balanced = balance_data(X_train_scaled, y_train)
        
        # Free more memory
        del X_train, X_train_scaled
        gc.collect()
        
        # Train model
        model = train_model(X_train_balanced, y_train_balanced, X_test_scaled, y_test)
        
        # Evaluate model
        accuracy, f1, conf_matrix, class_report = evaluate_model(model, X_test_scaled, y_test)
        
        # Visualize results
        visualize_results(model, X_test.columns)
        
        # Save model
        model_path = save_model(model, 'xgboost_netguardian_large_data.pkl')
        
        # Training summary
        total_time = time.time() - start_time
        logger.info(f"Training pipeline completed in {total_time:.2f} seconds")
        logger.info(f"Final model accuracy: {accuracy:.4f}, F1 score: {f1:.4f}")
        logger.info(f"Model saved to {model_path}")
        
    except Exception as e:
        logger.error(f"Error in training pipeline: {str(e)}", exc_info=True)
        raise
        
if __name__ == "__main__":
    main()

100%|██████████| 2830743/2830743 [00:26<00:00, 108844.48it/s]


Fitting 3 folds for each of 10 candidates, totalling 30 fits


/usr/local/lib/python3.10/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


Accuracy: 0.9988
F1 Score: 0.9970
Confusion Matrix:
[[453672    593]
 [    76 111235]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    454265
           1       0.99      1.00      1.00    111311

    accuracy                           1.00    565576
   macro avg       1.00      1.00      1.00    565576
weighted avg       1.00      1.00      1.00    565576



/usr/local/lib/python3.10/dist-packages/xgboost/core.py:160: UserWarning: [17:41:39] WARNING: /workspace/src/c_api/c_api.cc:1240: Saving into deprecated binary model format, please consider using `json` or `ubj`. Model format will default to JSON in XGBoost 2.2 if not specified.
  warnings.warn(smsg, UserWarning)


<Figure size 1200x800 with 0 Axes>

<Figure size 1200x800 with 0 Axes>